# Linear Regression

---
This notebook shows how to train, test and check a linear regression model in R. It moves from a synthetic straight line, to a model with one real regressor, to a model with many regressors on bank marketing data, and finally to a forecasting model on Bitcoin returns.

**Learning objectives.** By the end of this notebook you should be able to:

1. Fit a linear regression with one regressor and read its two numbers (intercept and slope) as a plain business statement.
2. Build a model with many regressors from real data using only information that is known at decision time, and explain why anything observed later must be left out.
3. Judge a model on a test set by RMSE and R², compare both with a "predict the mean" baseline, and say what a low R² does and does not mean.
4. Check the five regression assumptions with the right plot or statistic for each, including a variance inflation factor (VIF) check for multicollinearity.
5. Explain why a linear model on lagged Bitcoin returns has almost no forecasting skill, and why regressing a price on yesterday's price gives an R² near 1 that means nothing.

**Data used.** `Admission_Predict.csv` (one regressor), `banking.csv` (many regressors) and `data_BTC.csv` (a daily time series). All three are read directly from the course repository on GitHub, so nothing has to be downloaded by hand.

**Note on loading libraries:**

General syntax to load a library:
*library(library_name)*, e.g. *library(readr)*

A function from a library that is not loaded can be called with the library name in front:
*library_name::function_name()*, e.g. *car::vif(model)*

**Libraries:**

**readr** -- a fast and friendly way to read rectangular data (csv files), also directly from a URL.

**corrplot** -- a visual tool for correlation matrices.

**car** -- a collection of functions for applied regression; we use it for the variance inflation factors.

**lmtest** -- diagnostic tests for linear regression models; we use it for the Durbin-Watson test.

**glmnet** -- Ridge and Lasso regression (used in Exercise 7).

Everything else (`lm`, `predict`, `plot`, `acf`, ...) is base R.


In [ ]:
# Setup: install missing packages as pre-built binaries from Posit Package Manager.
# Colab's R runtime is Ubuntu, and compiling packages from source there takes 15 to 30 minutes; the binaries take under a minute.
if (Sys.info()[["sysname"]] == "Linux") {
  os_release <- tryCatch(readLines("/etc/os-release"), error = function(e) character(0))
  codename <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os_release, value = TRUE))
  if (!length(codename) || !nzchar(codename)) codename <- "jammy"   # Ubuntu 22.04, the current Colab runtime
  options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)))
} else {
  options(repos = c(CRAN = "https://cloud.r-project.org"))
}

# Optional: keep installed packages on Google Drive so they survive a runtime restart (only if Drive is mounted)
if (dir.exists("/content/drive/MyDrive")) {
  drive_lib <- "/content/drive/MyDrive/R/libs"
  dir.create(drive_lib, recursive = TRUE, showWarnings = FALSE)
  .libPaths(c(drive_lib, .libPaths()))
}

# Only the packages this notebook actually uses
required_packages <- c("readr", "corrplot", "car", "lmtest", "glmnet")
missing_packages <- setdiff(required_packages, rownames(installed.packages()))
if (length(missing_packages) > 0) install.packages(missing_packages)
for (pkg in required_packages) library(pkg, character.only = TRUE)

cat("All required packages are installed and loaded.\n")


In [ ]:
# To make this notebook's output stable across runs (we make the output reproducible)
set.seed(42)


In [ ]:
# How to get help on a function: put a question mark in front of its name
?print


## A synthetic straight line

We start with data that we generate ourselves, so we know the true relationship: y = 4 + 3 x + noise. This lets us see what a linear model is supposed to recover before we move to real data, where the truth is unknown.


In [ ]:
# Let's generate some linear looking data:
# Note: runif generates samples from the uniform distribution, while rnorm from the normal
X <- 2 * runif(100, 0, 1)


In [ ]:
y <- 4 + 3 * X + rnorm(100, 0, 1) # notice a difference between the function to generate X and y? The former draws from a uniform distribution and the latter from a normal distribution.


In [ ]:
print(cbind(X, y))  # cbind combines vectors by columns


In [ ]:
# Let's plot (info on the marker and the color --> see ?plot and ?par for details)
plot(X, y, pch=16, col="blue", xlab="x", ylab="y", xlim=c(0, 2), ylim=c(0, 12))


In [ ]:
# Training a linear model
lin_reg <- lm(y ~ X) # create and fit the linear regression
Y_predict <- predict(lin_reg, data.frame(X = X))
cat('Intercept:', round(coef(lin_reg)[1], 2), ' Slope:', round(coef(lin_reg)[2], 2), ' (true values: 4 and 3)\n')


In [ ]:
plot(X, y, pch=16, col="blue")
lines(X[order(X)], Y_predict[order(X)], col="red", lwd=2)


In [ ]:
X_new <- c(0.5, 1.75)
y_predict <- predict(lin_reg, data.frame(X = X_new))
y_predict


## One real regressor: GRE score and chance of admission

---

`Admission_Predict.csv` has one row per applicant to a graduate programme and two columns:

- `GRE_score`: the applicant's score on the GRE test (290 to 340 in this file)
- `Admit`: the estimated chance of admission, a number between 0 and 1

With a single regressor the fitted model is a line, y = intercept + slope * x, and the slope has a direct reading: "one more GRE point is associated with `slope` more chance of admission". This is the last time in the notebook that the whole model fits into one sentence.


In [ ]:
admission_url <- "https://raw.githubusercontent.com/umatter/EDFB/main/data/Admission_Predict.csv"
admission <- as.data.frame(read_csv(admission_url, show_col_types = FALSE))
print(dim(admission))
summary(admission)


In [ ]:
# Scatter plot: is a straight line a reasonable description?
plot(admission$GRE_score, admission$Admit, pch=16, cex=0.6,
     xlab='GRE score', ylab='Chance of admission')


In [ ]:
# Fit the univariate model: Admit = intercept + slope * GRE_score
model_adm <- lm(Admit ~ GRE_score, data = admission)

cat('Intercept:', coef(model_adm)[1], '\n')
cat('Slope:', coef(model_adm)[2], '\n')
cat('\nThe fitted model is Admit =', round(coef(model_adm)[1], 3), '+', round(coef(model_adm)[2], 4), '* GRE_score\n')
cat('R-squared (in sample):', round(summary(model_adm)$r.squared, 3), '\n')


In [ ]:
# Plot data and fitted line
gre_grid <- data.frame(GRE_score = seq(290, 340, length.out = 50))
plot(admission$GRE_score, admission$Admit, pch=16, cex=0.6, col="gray",
     xlab='GRE score', ylab='Chance of admission')
lines(gre_grid$GRE_score, predict(model_adm, newdata = gre_grid), col="red", lwd=2)


**Reading the result.** In this run the slope is about 0.010: ten more GRE points go with roughly 0.10 (ten percentage points) more chance of admission. The intercept (about -2.4) is the predicted chance at a GRE score of 0, a value that does not occur, so it has no meaning on its own; it only positions the line. R² is about 0.64: GRE score alone accounts for roughly two thirds of the variation in admission chances across applicants.

**Assumption 2 (multicollinearity) for this model:** with a single regressor there is nothing for it to be collinear with, so the check is empty here. It becomes important in the next model, which has more than twenty regressors.


## Many regressors: predicting call duration in a bank's marketing campaign

---

**The data.** `banking.csv` is the bank marketing dataset from a Portuguese bank. One row is **one phone call** made during a marketing campaign for a term deposit (41,188 calls). The columns:

| column | meaning |
|---|---|
| `age`, `marital`, `education`, `housing`, `loan` | client characteristics (housing/personal loan: yes / no / unknown) |
| `contact` | how the client was reached: cellular or telephone (landline) |
| `previous` | number of contacts with this client before the current campaign |
| `pdays` | days since the client was last contacted in a *previous* campaign; **999 means never contacted before** (96% of rows) |
| `poutcome` | outcome of the previous campaign: success, failure, or nonexistent |
| `emp_var_rate`, `cons_price_idx`, `cons_conf_idx`, `euribor3m`, `nr_employed` | macroeconomic indicators at the time of the call: employment variation rate, consumer price index, consumer confidence index, 3-month Euribor rate, number of employees in the economy (thousands) |
| `campaign` | number of contacts during this campaign (including this call) |
| `duration` | length of the call **in seconds** |
| `y` | did the client subscribe to the term deposit? |

**The business question.** A call centre plans its staffing from the expected length of calls. Can we predict how long a call will last from what is known *before* the agent dials?

**The decision-time leakage rule.** Ask: *at the moment the decision is made, which columns are already known?* Before the call, the client's characteristics, the contact history and the macro indicators are known. `duration` is the target, `y` is only known at the end of the call, and `campaign` counts the current call itself, so these three are excluded from the regressors. Anything we would only learn during or after the call may not be used to predict it.

**Why `log1p(duration)`.** Call durations are heavily right-skewed (median 180 seconds, maximum 4,918 seconds). Modelling log(1 + duration) keeps the few very long calls from dominating the fit, and turns coefficients into approximate percentage effects: a coefficient of 0.10 means roughly 10% longer calls. `log1p` rather than `log` because a few calls have duration 0.

**`pdays = 999`.** The value 999 is a code for "never contacted", not a number of days. We split it into a flag `was_previously_contacted` and a cleaned `pdays_clean` in which the code is replaced by the median of the real values. That median is computed on the full data for simplicity; in a deployed pipeline it would be computed on the training rows only, so that nothing from the test rows enters the features.


In [ ]:
banking_url <- "https://raw.githubusercontent.com/umatter/EDFB/main/data/banking.csv"
print("Fetching banking.csv from GitHub...")


In [ ]:
dataset <- as.data.frame(read_csv(banking_url, show_col_types = FALSE))


In [ ]:
head(dataset)


In [ ]:
tail(dataset)


In [ ]:
dim(dataset) # Returns the dimensions of the data frame


In [ ]:
str(dataset) # Returns the structure of the data frame


In [ ]:
# Check for NAs
sapply(dataset, function(x) any(is.na(x))) # Check if any column has missing values


In [ ]:
sapply(dataset, function(x) sum(is.na(x))) # Count missing values per column


In [ ]:
# Describe the data
summary(dataset)


In [ ]:
print(colnames(dataset))


In [ ]:
# Select the target and the regressors: predict log1p(duration) from pre-call client, contact-history and macro features.
# Leakage rule: 'duration' is the target, 'y' and 'campaign' are only known during or after the call, so they are excluded.
df_banking <- dataset
df_banking$was_previously_contacted <- as.numeric(df_banking$pdays != 999)
df_banking$pdays_clean <- ifelse(df_banking$pdays == 999, NA, df_banking$pdays)
df_banking$pdays_clean[is.na(df_banking$pdays_clean)] <- median(df_banking$pdays_clean, na.rm = TRUE)   # median of the full data (simplification, see text)
df_banking$log_duration <- log1p(df_banking$duration)

feature_cols_cat <- c('marital', 'education', 'housing', 'loan', 'contact', 'poutcome')
feature_cols_num <- c('age', 'previous', 'pdays_clean', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'was_previously_contacted')
for (col in feature_cols_cat) df_banking[[col]] <- factor(df_banking[[col]])

# One dummy column per category level, dropping the first level of each variable (the reference category).
# model.matrix adds an intercept column, which we drop; lm() adds its own intercept later.
X_df <- model.matrix(~ ., data = df_banking[, c(feature_cols_cat, feature_cols_num)])[, -1]

# 'housing' and 'loan' are 'unknown' for exactly the same 990 clients, so the two 'unknown' dummies are
# identical columns. Two identical regressors cannot both get a coefficient (perfect multicollinearity), so keep one.
X_df <- as.data.frame(X_df[, colnames(X_df) != "loanunknown"])

y <- df_banking$log_duration
cat('Regressors:', ncol(X_df), '\n')
print(colnames(X_df))

# Choose one feature for visualisation, by name
viz_col <- 'age'

# Preserve banking variables so they are not overwritten by the Bitcoin section later on
X_df_banking <- X_df
y_banking <- y
viz_col_banking <- viz_col


In [ ]:
# Scatter plot of the target against one feature (age)
plot(X_df_banking[[viz_col_banking]], y_banking, pch=16, cex=0.3, col=rgb(0, 0, 0, 0.2),
     xlab=viz_col_banking, ylab='log1p(duration)')


In [ ]:
# Correlations between the numeric variables (raw dataset, categorical columns and the outcome y removed)
numeric_cols <- setdiff(colnames(dataset)[sapply(dataset, is.numeric)], 'y')
corrmat <- cor(dataset[, numeric_cols])
round(corrmat, 2)


In [ ]:
# Plot correlation heatmap
corrplot(corrmat, method = "color", type = "upper", addCoef.col = "black", number.cex = 0.7,
         tl.cex = 0.8, tl.col = "black", tl.srt = 45)


The heatmap already shows one thing to remember for the assumption checks below: the macro indicators `emp_var_rate`, `euribor3m` and `nr_employed` are correlated at 0.9 or more with each other. They all measure the state of the economy at the time of the call, so they move together.


In [ ]:
# Split train and test set (20% in testing). We set the seed, as every time you run it without a seed you get a different split.
df_model <- data.frame(X_df, log_duration = y)
set.seed(0)
train_idx <- sample(seq_len(nrow(df_model)), size = floor(0.8 * nrow(df_model)))
train_banking <- df_model[train_idx, ]
test_banking <- df_model[-train_idx, ]
X_train <- train_banking[, colnames(X_df)]
X_test <- test_banking[, colnames(X_df)]
y_train <- train_banking$log_duration
y_test <- test_banking$log_duration

print(dim(X_train))
print(dim(X_test))
print(length(y_train))
print(length(y_test))

# Preserve banking split to avoid being overwritten by later BTC section
X_train_banking <- X_train
X_test_banking <- X_test
y_train_banking <- y_train
y_test_banking <- y_test


In [ ]:
# Fit the model on training set. 'log_duration ~ .' means: regress log_duration on every other column of the data frame.
model <- lm(log_duration ~ ., data = train_banking) # training the algorithm


In [ ]:
# Get coefficients: one per regressor, so print them as a labelled table
cat('Intercept:', round(coef(model)[1], 3), '\n')
coef_table <- round(coef(model)[-1], 4)
print(coef_table)


There is no single "slope" any more: each coefficient is the change in log1p(duration) for a one-unit change in *that* regressor, holding the others fixed. For a dummy column the unit is "belongs to this category rather than the reference category". For example, in this run `contacttelephone` is roughly -0.2: calls to a landline are roughly 20% shorter than calls to a mobile, other things equal. Whether a coefficient is also *precisely estimated* is the subject of Exercise 1.


In [ ]:
# Get fitted value on test set
y_test_predicted <- predict(model, newdata = test_banking)
y_test_predicted_banking <- y_test_predicted

# Compare predictions
comparison_df <- data.frame(True = y_test, Predicted = y_test_predicted)
head(comparison_df, 10)


In [ ]:
# Plot predicted against true values on the test set. A perfect model would put every point on the dashed 45-degree line.
plot(y_test, y_test_predicted, pch=16, cex=0.3, col=rgb(0.5, 0.5, 0.5, 0.2),
     xlab='True log1p(duration)', ylab='Predicted log1p(duration)')
abline(a=0, b=1, lty=2, lwd=2)
legend("topleft", legend="45-degree line (perfect prediction)", lty=2, lwd=2, bty="n")


In [ ]:
# Plot some predicted vs true values against the visualisation feature (first 30 test observations)
points_to_plot <- 30
x_subset <- X_test[[viz_col_banking]][1:points_to_plot]
plot(x_subset, y_test[1:points_to_plot], pch=1, col="blue",
     xlab=viz_col_banking, ylab="log1p(duration)", main="True vs Predicted")
points(x_subset, y_test_predicted[1:points_to_plot], pch=4, col="red")
legend("topright", legend=c("true value", "predicted"), pch=c(1, 4), col=c("blue", "red"))


In [ ]:
# Evaluate Root Mean Square Error (RMSE), next to the scale of the target and next to a mean-only baseline
RMSE_test <- sqrt(mean((y_test - y_test_predicted)^2))
RMSE_mean_only <- sqrt(mean((y_test - mean(y_train))^2))   # always predict the training mean
cat('Root Mean Squared Error on test set:', round(RMSE_test, 4), '\n')
cat('RMSE of predicting the training mean:', round(RMSE_mean_only, 4), '\n')
cat('Mean of log1p(duration) in y_test:', round(mean(y_test), 4), '\n')
cat('Standard deviation of log1p(duration) in y_test:', round(sd(y_test), 4), '\n')


In [ ]:
# Evaluate R-squared on the test set: 1 - SSE / SST (the out-of-sample R-squared, same definition as sklearn's r2_score)
r2_oos <- function(y, y_hat) 1 - sum((y - y_hat)^2) / sum((y - mean(y))^2)
R2 <- r2_oos(y_test, y_test_predicted)
cat('R-squared on test set:', round(R2, 4), '\n')
cat('R-squared on training set:', round(summary(model)$r.squared, 4), '\n')


**What these numbers say.** In this run the test R² is about 0.01: the model explains roughly 1% of the variation in log call duration. The RMSE (about 0.91) is within one percent of the standard deviation of the target, and only a hair below the RMSE of a model that always predicts the training mean. On the log scale an error of 0.9 means the typical prediction is off by a factor of about 2.5 in either direction.

**Why so low, and why that is normal.** How long a call lasts depends mostly on what happens *during* the call: whether the client is interested, asks questions, or hangs up. None of that is known before dialling, and the leakage rule says we may not use it. Pre-call characteristics carry only a little information about call length. A low R² here is not a bug in the code; it is the honest answer to the question "how predictable is call length from what we know in advance?", and that answer ("hardly at all") is itself useful for the call centre: it should plan on the average with a wide margin rather than trust per-call predictions.

**Two things a low R² does not mean.** It does not mean the coefficients are wrong or unimportant: with 33,000 training calls, several of them are estimated precisely (Exercise 1), and they tell us which client and contact features shift call length on average. And it does not mean a different model would do much better; Exercise 3 and Exercise 7 will show that adding features or regularising changes little, because the missing information is not in the data.

Note that `cor(y_test, y_test_predicted)^2`, which is sometimes used as "R²", is a different quantity: it is always at least zero and ignores whether the predictions are biased or on the wrong scale. The 1 - SSE/SST definition used here can be negative on a test set (see the Bitcoin section).


## Check linear regression assumptions

**Linear regression assumes the following:**

---

1. a **linear relationship** between each regressor and the target
2. **no perfect multicollinearity** between regressors, and not too much strong multicollinearity
3. **homoscedasticity**: the variance of the error terms (residuals) does not depend on the fitted value or on the regressors
4. **normally distributed error terms** (this matters for confidence intervals and p-values in small samples, not for the predictions themselves)
5. **independent error terms**: for time series data this means **no autocorrelation** in the residuals, i.e. no correlation between $e_t$ and $e_{t-1}$

You will sometimes see "no correlation between regressors and residuals" (exogeneity) listed as an assumption. It cannot be checked from the residuals of the fitted model: on the training data, ordinary least squares makes those correlations exactly zero by construction, so the check would always pass. Exogeneity is a statement about how the data came about (no omitted variables that move with the regressors) and has to be argued, not plotted.

Assumptions 1 to 4 are checked on the banking model here. Assumption 5 is a statement about the *order* of observations; the banking rows are individual calls in arbitrary order, so there is nothing to check. We check it on the Bitcoin model below, where the order is time.


In [ ]:
# Checking Assumption 1 - linear relationship between a regressor and the target
# With 41,000 calls a raw scatter is a solid block, so plot the mean of log1p(duration) at each age.
mean_by_age <- aggregate(log_duration ~ age, data = df_banking, FUN = mean)
plot(X_df_banking[[viz_col_banking]], y_banking, pch=16, cex=0.3, col=rgb(0.5, 0.5, 0.5, 0.05),
     main='Feature vs log1p(duration)', xlab=viz_col_banking, ylab='log1p(duration)')
lines(mean_by_age$age, mean_by_age$log_duration, col="red", lwd=2)
legend("topright", legend=c("individual calls", "mean log1p(duration) at each age"),
       pch=c(16, NA), lty=c(NA, 1), col=c("gray", "red"), lwd=c(NA, 2))


The mean of the target moves very little with age and roughly along a line; the few ages above 80 have only a handful of calls each, which is why the red line jumps around there. Nothing here calls for a curved term in age.

**Checking Assumption 2 - little to no multicollinearity between regressors**

Multicollinearity means that one regressor can be predicted well from the others. It does not hurt predictions, but it makes the individual coefficients unstable and their standard errors large: the model cannot tell which of two near-identical regressors deserves the credit. The variance inflation factor (VIF) of a regressor is 1 / (1 - R²) of the regression of that regressor on all the others. VIF = 1 means no collinearity; values above 5 are worth noting and above 10 are usually called high.


In [ ]:
# Checking Assumption 2 - VIF for every regressor of the banking model (car::vif works directly on the fitted lm object)
vif_values <- sort(car::vif(model), decreasing = TRUE)
print(round(vif_values, 1))


In this run the three macro indicators have VIFs between about 30 and 65 and `emp_var_rate`, `euribor3m` and `nr_employed` are the culprits: as the heatmap showed, they measure the same thing (the state of the economy) and move together. `was_previously_contacted` and `poutcomesuccess` are also collinear (VIF around 13 to 14), because a previous campaign can only have succeeded for clients who were contacted before. Everything else is below 10.

What follows from that: the predictions are fine, but the *individual* coefficients of the three macro variables should not be read separately. A safe reading is "a better economy (higher rates, more employment) goes with shorter calls"; splitting that into "Euribor does X and employment does Y" is not supported. If we had left both `unknown` dummies in the model, `lm` would have reported one coefficient as `NA` (perfect multicollinearity) and `vif` would have refused to run; that is why the construction cell dropped `loanunknown`.


In [ ]:
# Checking Assumption 3 - Homoscedasticity
# Plot the residuals against the fitted values. Under homoscedasticity the vertical spread of the points is the same
# for small and large fitted values. A funnel (narrow on one side, wide on the other) indicates heteroscedasticity.
residuals_test <- y_test_banking - y_test_predicted_banking
plot(y_test_predicted_banking, residuals_test, pch=1, cex=0.5, col=rgb(0, 0, 0, 0.3),
     xlab="Fitted values (predicted log1p(duration))", ylab="Residuals (true - predicted)",
     main="Residuals vs fitted values")
abline(h=0, col="red", lty=2)


The spread of the residuals is about the same across the range of fitted values, so there is no funnel. Note the diagonal edge at the bottom left: it comes from the few calls with duration 0, whose residual is 0 minus the fitted value.


In [ ]:
# Checking Assumption 4 - Normal distribution of residuals
# Check if the residual distribution looks like a normal distribution with the same mean and variance

resid_mean <- mean(residuals_test)
resid_std <- sd(residuals_test)
normal_distr <- rnorm(length(residuals_test), resid_mean, resid_std)

# Create three plots
par(mfrow=c(1,3))

# Plot 1: Residual distribution
hist(residuals_test, main='Residual distribution', xlab='Residuals', breaks=40, col='lightblue', freq=FALSE)

# Plot 2: Normal distribution
hist(normal_distr, main='Normal distribution', xlab='Values', breaks=40, col='orange', freq=FALSE)

# Plot 3: Overlay
hist(residuals_test, main='Comparison', xlab='Values', breaks=40, col=rgb(0,0,1,0.5), freq=FALSE)
hist(normal_distr, add=TRUE, breaks=40, col=rgb(1,0.5,0,0.5), freq=FALSE)
legend("topleft", legend=c("residuals", "normal distribution"), fill=c(rgb(0,0,1,0.5), rgb(1,0.5,0,0.5)))

par(mfrow=c(1,1))


In [ ]:
# Check Q-Q plot: the quantiles of the residuals against the theoretical quantiles of a normal distribution.
# If the residuals were normal, the points would lie on the dashed line (qqline passes through the first and third quartiles).
qqnorm(residuals_test, pch=1, col="blue", xlab='theoretical normal quantiles', ylab='residual quantiles', main='Normal Q-Q plot of the residuals')
qqline(residuals_test, lty=2, col="black")


The residuals are close to normal in the centre and have a heavier left tail (the very short calls). With 8,000 test observations this does not affect the predictions or the coefficient estimates; it would only matter for confidence intervals in a very small sample.

**Assumption 5 (independence, no autocorrelation)** is checked in the Bitcoin section, where the observations have a time order.


## Linear regression for forecasting: Bitcoin

---

In this section we use a linear model to **forecast** something: tomorrow's Bitcoin return from the returns of the last five days. The data is `data_BTC.csv`, one row per day with the closing price in US dollars (`BTC-USD.Close`) from August 2017 to September 2026.

Two ideas make this section different from the banking model:

1. **The order of the rows is time.** We may only use the past to predict the future, so the train/test split is chronological (first 80% of days for training, last 20% for testing), and every regressor for day *t* must be something that was known at the end of day *t - 1*.
2. **We forecast returns, not prices.** The first model below shows why: regressing today's price on yesterday's price gives an R² near 1 that means nothing.


In [ ]:
# Let's import the dataset including Bitcoin prices
btc_url <- "https://raw.githubusercontent.com/umatter/EDFB/main/data/data_BTC.csv"
print("Fetching data_BTC.csv from GitHub...")


In [ ]:
data <- as.data.frame(read_csv(btc_url, show_col_types = FALSE))
data$Date <- as.Date(data$Date)
# Ensure strict chronological order
data <- data[order(data$Date), ]
rownames(data) <- NULL


In [ ]:
# Let's check if we imported correctly
head(data)


In [ ]:
# Let's get some summary stats on the prices
summary(data)


In [ ]:
# Let's plot the movement of the BTC Close Price
plot(data$Date, data$`BTC-USD.Close`, type='l', col='blue',
     xlab='Date', ylab='Price (USD)', main='Bitcoin Price Over Time')


### The trap: regressing price on lagged price

The obvious first idea is to predict today's price from yesterday's price. Let us do that, and also compare it with the simplest possible "forecast": just say that today's price equals yesterday's price.


In [ ]:
# Price on lagged price, chronological split
price <- data$`BTC-USD.Close`
price_df <- data.frame(price = price, price_lag_1 = c(NA, price[-length(price)]))
price_df <- price_df[complete.cases(price_df), ]
split_p <- floor(nrow(price_df) * 0.8)
train_p <- price_df[1:split_p, ]
test_p <- price_df[(split_p + 1):nrow(price_df), ]

model_price <- lm(price ~ price_lag_1, data = train_p)
pred_price <- predict(model_price, newdata = test_p)

cat('Fitted model: price_t =', round(coef(model_price)[1], 2), '+', round(coef(model_price)[2], 4), '* price_{t-1}\n')
cat('R-squared on test set (model):        ', round(r2_oos(test_p$price, pred_price), 4), '\n')
cat('R-squared on test set ("price_t = price_{t-1}", no model at all):', round(r2_oos(test_p$price, test_p$price_lag_1), 4), '\n')


In this run the model has a test R² of about 0.99, and its slope is about 1.00. But "tomorrow's price is today's price", which requires no model and no data, has the *same* R². The high R² comes entirely from the fact that prices wander slowly: tomorrow's price is always close to today's, whatever happens next. The model has learned nothing about *where* the price is going. A price series is **non-stationary** (its level drifts and its variance grows over time), and a regression on a non-stationary series produces exactly this kind of impressive-looking, useless fit.

### Forecasting returns instead

What a trader actually needs is the *change*: the **log return** $r_t = \log(P_t) - \log(P_{t-1})$, roughly the percentage change of the price. Returns are close to stationary (they hover around zero with a roughly stable spread), so a regression of $r_t$ on past returns $r_{t-1}, \dots, r_{t-5}$ is a fair test of whether the past contains information about the future. Set your expectation now: if markets are even roughly efficient, the answer will be "almost none", and the test R² will be around zero. That is the honest result to look for, not a failure of the method.


In [ ]:
# Create lagged return features. This function does not mutate its input data frame.
create_lagged_features <- function(data, lag) {
  df <- data
  df$ret <- c(NA, diff(log(df$`BTC-USD.Close`)))      # log return of day t
  for (i in 1:lag) {
    df[[paste0('lag_ret_', i)]] <- c(rep(NA, i), df$ret[1:(nrow(df) - i)])   # the return i days earlier: known at the end of day t-1
  }
  df <- df[complete.cases(df), ]
  rownames(df) <- NULL
  return(df)
}


In [ ]:
# Create lag features on log returns with 5 lags
data <- create_lagged_features(data, lag=5)


In [ ]:
head(data)


**Which columns may be regressors?** Only the five lagged returns. `ret` is the target. `BTC-USD.Close` is the closing price of day *t* itself, which is not known until the day is over; using it (or any open, high, low or volume figure of the same day) would let the model see the answer. The rule is the same as in the banking model: nothing observed during period *t* may be used to predict period *t*.


In [ ]:
# Split the data into training and testing sets using a chronological split (avoid leakage)
feature_cols_btc <- paste0('lag_ret_', 1:5)
X_btc <- data[, feature_cols_btc]
y_btc <- data$ret
split_idx <- floor(nrow(data) * 0.8)
X_train_btc <- X_btc[1:split_idx, ]
X_test_btc <- X_btc[(split_idx + 1):nrow(X_btc), ]
y_train_btc <- y_btc[1:split_idx]
y_test_btc <- y_btc[(split_idx + 1):length(y_btc)]
cat('Training days:', nrow(X_train_btc), ' Test days:', nrow(X_test_btc), ' Test period starts:', format(data$Date[split_idx + 1]), '\n')


In [ ]:
# Train a linear regression model (lm adds the intercept itself) and print the summary with standard errors and p-values
train_btc <- data.frame(X_train_btc, ret = y_train_btc)
model_btc <- lm(ret ~ ., data = train_btc)

# Print the summary, which includes coefficients and p-values
summary(model_btc)


**How to read this table.** The `Coefficients` block has one row per regressor:

- `Estimate` is the estimated coefficient, as before.
- `Std. Error` is the standard error: how much the coefficient would wobble if we re-estimated it on a different sample of days. A coefficient is *precisely estimated* when it is several times its standard error.
- `t value` is Estimate / Std. Error, and `Pr(>|t|)` is the p-value: the probability of seeing a coefficient at least this far from zero if the true coefficient were zero. A small p-value (say below 0.05) is evidence that the coefficient is not exactly zero. It is **not** a measure of importance: with thousands of observations even a tiny effect gets a small p-value.
- A 95% confidence interval is roughly Estimate plus or minus 2 standard errors (`confint(model_btc)` prints it).

Below the block, `Multiple R-squared` is the in-sample R², here about 0.006 in this run: the five lags explain about half a percent of the variation of daily returns. In this run `lag_ret_1` and `lag_ret_2` have p-values around 0.01 to 0.02, so there is weak statistical evidence of a small negative first-order and positive second-order dependence. Statistically detectable and economically useful are different things: a "significant" lag inside an R² of 0.006 is of no use for trading, as the test set will show.


In [ ]:
# Make predictions on the test set
y_pred_btc <- predict(model_btc, newdata = X_test_btc)

# Root Mean Squared Error of the model
rmse <- sqrt(mean((y_test_btc - y_pred_btc)^2))

# Benchmark: the zero forecast ("no change", tomorrow's expected return is 0). A model with forecasting skill must beat it.
rmse_zero <- sqrt(mean(y_test_btc^2))

# A tempting but wrong benchmark: predict tomorrow's return with today's return (lag 1).
# Returns are close to noise, so this benchmark doubles the error variance and makes any model look 30% better than it is.
rmse_lag1 <- sqrt(mean((y_test_btc - X_test_btc$lag_ret_1)^2))

# R-squared on the test set: 1 - SSE / SST
r2 <- r2_oos(y_test_btc, y_pred_btc)

cat(sprintf("RMSE (model):                    %.6f\n", rmse))
cat(sprintf("RMSE (zero forecast):            %.6f\n", rmse_zero))
cat(sprintf("RMSE ratio (model / zero):       %.3f   (below 1 would mean forecasting skill)\n", rmse / rmse_zero))
cat(sprintf("RMSE (lag-1 'benchmark'):        %.6f\n", rmse_lag1))
cat(sprintf("RMSE ratio (model / lag-1):      %.3f   (looks good, but only because the benchmark is bad)\n", rmse / rmse_lag1))
cat(sprintf("R-squared on test set:           %.4f\n", r2))
cat(sprintf("Standard deviation of test returns: %.6f\n", sd(y_test_btc)))


**Reading the result.** In this run the model's RMSE is 1.003 times the RMSE of the zero forecast and the test R² is about -0.006. A model that cannot beat "no change" has no forecasting skill, and this one cannot. Against the lag-1 "benchmark" the ratio is about 0.69, which looks like a 30% improvement; it is not, because predicting a noisy series by its own last value is a much worse forecast than predicting zero (its error variance is twice as large). Always compare a return forecast with the zero or the training-mean forecast.

**Why can test R² be negative?** R² on a test set is 1 minus (sum of squared errors of the model) divided by (sum of squared deviations from the test mean). Nothing forces the model to do better than the test mean on data it has not seen, and when it does worse, R² goes below zero. A test R² of -0.006 means the model is a little worse than a flat line at the test-set average. Squared correlation (`cor(y, y_hat)^2`), which is sometimes used instead, is always at least zero and would hide this.


In [ ]:
# A few predictions next to the true values (daily returns are around 0.02 in size, so use 4 decimals)
head(data.frame(Date = data$Date[(split_idx + 1):nrow(data)],
                True = round(y_test_btc, 4),
                Predicted = round(y_pred_btc, 4)), 10)


In [ ]:
# Plot true vs predicted returns over time. The predictions are a nearly flat line: the model has almost nothing to say.
test_dates <- data$Date[(split_idx + 1):nrow(data)]
plot(test_dates, y_test_btc, type='l', col='blue', xlab='Date', ylab='Log return', main='True vs Predicted Returns')
lines(test_dates, y_pred_btc, col='red')
legend("topright", legend=c("True returns", "Predicted returns"), col=c("blue", "red"), lty=1)


### Checking Assumption 5 on the Bitcoin model: no autocorrelation in the residuals

Here the observations are ordered in time, so the assumption has content. If the residuals of day *t* were correlated with those of day *t - 1*, the model would be leaving predictable structure on the table. Two standard checks:

- the **Durbin-Watson statistic** $DW = \sum (e_t - e_{t-1})^2 / \sum e_t^2$, which is 2 when there is no first-order autocorrelation, below 2 for positive and above 2 for negative autocorrelation;
- the **autocorrelation function (ACF)** of the residuals: the correlation of $e_t$ with $e_{t-k}$ for k = 1, 2, ..., plotted with a band outside which a correlation would be more than noise.


In [ ]:
resid_btc <- resid(model_btc)
dw <- lmtest::dwtest(model_btc)
cat('Durbin-Watson statistic (training residuals):', round(dw$statistic, 3), ' (2 = no first-order autocorrelation), p-value:', round(dw$p.value, 3), '\n')

par(mfrow=c(1,2))
acf(y_train_btc, lag.max=20, main='ACF of the returns themselves')
acf(resid_btc, lag.max=20, main='ACF of the model residuals')
par(mfrow=c(1,1))


In this run the Durbin-Watson statistic is 2.00 and the residual autocorrelations are all inside the noise band. The left panel shows why the model had so little to work with: the returns themselves are almost uncorrelated from one day to the next (the lag-1 and lag-2 correlations are around 0.05 in size), and the model has absorbed that small amount. What is left is white noise, which is exactly what weak-form market efficiency predicts.


## Exercises

Now it's time to practice. Exercises marked **core** are the ones to do in class; **optional** ones go deeper. Each exercise names the deliverable to hand in.

### Exercise 1 (core): Which coefficients are large and precisely estimated?

**Task:** Interpret the coefficients of the banking model. A coefficient is worth a business sentence only if it is both large enough to matter *and* estimated precisely enough to trust.

**Instructions:**
1. Use `summary(model)$coefficients` (estimate, standard error, t value, p-value) and `confint(model)` (95% confidence intervals) of the banking model.
2. List the coefficients sorted by absolute size, together with their standard error and 95% confidence interval.
3. Pick the five coefficients that are both large (say, |coef| > 0.05, i.e. more than a 5% effect on call duration) and precisely estimated (confidence interval does not contain 0). Which large coefficients fail the second test?
4. Explain what the five mean in business terms (how they shift call duration), remembering the VIF result: the macro variables cannot be read one by one.

**Deliverable:** a table of the five coefficients with their confidence intervals and one sentence per coefficient on its business meaning.


In [ ]:
# Exercise 1: Your code here
# Hint: coef_summary <- summary(model)$coefficients; ci <- confint(model)
#       both are matrices with one row per regressor (the first row is the intercept)

# Your solution:


### Exercise 2 (core): Residual Analysis

**Task:** Perform a residual analysis of the banking model to check the assumptions.

**Instructions:**
1. Calculate residuals for both the training and the test set
2. Create a residuals vs fitted values plot
3. Check the normality of the residuals with a histogram and a Q-Q plot (`qqnorm`, `qqline`)
4. Look for patterns that would indicate an assumption violation

**Deliverable:** a 2 x 2 panel of diagnostic plots and one sentence per panel saying what it shows.


In [ ]:
# Exercise 2: Your code here
# Hint: Calculate residuals = actual - predicted
# Use par(mfrow=c(2,2)) to create multiple diagnostic plots

# Your solution:


### Exercise 3 (optional): Feature Engineering

**Task:** Create new features and see whether they improve the banking model.

**Business Context:** Sometimes combining existing features or creating polynomial terms can capture non-linear relationships.

**Instructions:**
1. Create interaction terms between two numerical features
2. Add polynomial features (squared terms) for numerical variables, and keep the dummy columns in the model so that the comparison is fair
3. Train a new model with these engineered features
4. Compare R² and RMSE on the test set with the original model

**Deliverable:** a table of test R² and RMSE for the original and the engineered model, and one sentence on whether the extra features helped.


In [ ]:
# Exercise 3: Your code here
# Hint: Use I(variable^2) for squared terms and variable1:variable2 for interactions in R formulas,
#       e.g. lm(log_duration ~ . + I(age^2) + age:previous, data = train_banking)
# Be careful about overfitting with too many features

# Your solution:


### Exercise 4 (core): Cross-Validation

**Task:** Use cross-validation to get a more robust estimate of model performance.

**Business Context:** Before deploying a model, you want to know that it performs consistently across different samples of the data.

**Instructions:**
1. Use 5-fold cross-validation on the banking dataset (shuffle the rows into folds)
2. Calculate the mean and the standard deviation of the R² scores across folds
3. Compare with a simple baseline model that predicts the mean of the training fold
4. Discuss the stability of your model

**Deliverable:** mean and standard deviation of R² across the five folds for the model and for the baseline, and one sentence on stability.


In [ ]:
# Exercise 4: Your code here
# Hint: set.seed(42); folds <- sample(rep(1:5, length.out = nrow(df_model)))
#       then loop over k in 1:5, fit on df_model[folds != k, ], evaluate r2_oos on df_model[folds == k, ]

# Your solution:


### Exercise 5 (core): Synthetic Data Generation

**Task:** Create your own synthetic dataset with known coefficients and see how well the model recovers them.

**Instructions:**
1. Generate a dataset with 500 observations and 3 features
2. Create a known linear relationship: y = 2*x1 + 3*x2 - 1.5*x3 + noise
3. Add some outliers to make it more realistic
4. Train a linear model and see how well it recovers the true coefficients
5. Experiment with different noise levels. Before you run it, predict: what happens to the coefficient estimates and to R² as the noise grows?

**Deliverable:** a table of true vs estimated coefficients at three noise levels, and one sentence comparing the outcome with your prediction.


In [ ]:
# Exercise 5: Your code here
# Hint: Use rnorm and runif functions to generate features and noise
# True coefficients should be [2, 3, -1.5]

set.seed(42)  # For reproducibility

# Your solution:


### Exercise 6 (core): Time Series Forecasting Analysis

**Task:** Analyse the Bitcoin forecasting model more deeply.

**Instructions:**
1. Calculate the directional accuracy (how often the model predicts the correct sign of the return) on the test set, and compare it with the share of days on which the return was positive (the accuracy of "always predict up")
2. Create a cumulative returns plot comparing a strategy that follows the model's sign with buy-and-hold
3. Test different lag lengths (1, 3, 5, 10). Choose the lag length on a validation slice of the *training* period (for example its last 25%), not on the test set, and only then report the test result of the chosen lag
4. Discuss the practical implications for trading, including transaction costs

**Deliverable:** the directional accuracy of the chosen model on the test set next to the share of up-days, and one sentence on whether the model has usable skill.


In [ ]:
# Exercise 6: Your code here
# Hint: Directional accuracy = mean(sign(predicted) == sign(actual))
# Cumulative returns = cumsum of returns over time

# Your solution:


### Exercise 7 (optional): Model Comparison

**Task:** Compare linear regression with Ridge and Lasso regression on the banking data.

**Instructions:**
1. Implement a Ridge regression model on the banking dataset (`glmnet` with `alpha = 0`)
2. Implement a Lasso regression model (`alpha = 1`)
3. Choose the regularisation strength `lambda` by cross-validation on the training data (`cv.glmnet`), never on the test set
4. Compare test R² and RMSE of the three models and discuss when you would prefer each

**Deliverable:** a table of test R² and RMSE for the three models with the chosen lambdas, and one sentence on which you would use here and why.


In [ ]:
# Exercise 7: Your code here
# Hint: glmnet needs matrices: x_train <- as.matrix(X_train_banking); glmnet standardises the columns itself
#       cv_ridge <- cv.glmnet(x_train, y_train_banking, alpha = 0); predict(cv_ridge, newx = as.matrix(X_test_banking), s = "lambda.min")

# Your solution:


### Exercise 8 (core): Business Impact Analysis

**Task:** Quantify the business value of the duration prediction model.

**Business Context:** Longer calls might indicate higher customer engagement and conversion probability. The target is log1p(duration) with `duration` in **seconds**, so the thresholds must be on that scale.

**Scenario:**
- Calls shorter than 2 minutes (log1p(duration) < log1p(120) ≈ 4.80): Low engagement
- Calls of 2 to 5 minutes (log1p(duration) between 4.80 and log1p(300) ≈ 5.71): Medium engagement
- Calls longer than 5 minutes (log1p(duration) > 5.71): High engagement
- You want to prioritise follow-up with predicted high-engagement calls

**Instructions:**
1. Classify actual and predicted test-set durations into the three engagement categories
2. Report the share of *actual* calls and the share of *predicted* calls in each category, and the confusion matrix
3. Calculate the accuracy for each engagement level
4. Estimate the business value of a "follow up only with predicted High" strategy against "follow up with everyone", under stated assumptions for cost per call and value per correctly identified high-engagement call

**Deliverable:** the share of test predictions in each band and one sentence on whether this model can be used to rank calls by engagement.


In [ ]:
# Exercise 8: Your code here
# Hint: Use cut() with breaks = c(-Inf, log1p(120), log1p(300), Inf) and labels = c("Low", "Medium", "High")
# Calculate the confusion matrix for the three categories with table()

# Your solution:


### Exercise 9 (optional): Advanced Diagnostics

**Task:** Perform advanced model diagnostics to identify potential issues.

**Instructions:**
1. Calculate Cook's distance for the banking model to identify influential observations. Never build the full hat matrix (33,000 x 33,000 would need 8 GB of memory): `cooks.distance(model)` works from the leverages `hatvalues(model)` and is fast; the closed form is `D = e^2 / (p * s2) * h / (1 - h)^2` with `e = resid(model)`, `p` the number of coefficients including the intercept and `s2 = summary(model)$sigma^2`.
2. Check for multicollinearity using the variance inflation factor (VIF)
3. Perform the Durbin-Watson test for autocorrelation on the residuals of the **Bitcoin** model, and explain why the statistic is meaningless on the banking model
4. Suggest improvements based on your findings

**Deliverable:** the number of influential observations, the regressors with VIF > 10, the Durbin-Watson statistic of the Bitcoin model, and one sentence on what each implies for the *inference* (not the predictions).


In [ ]:
# Exercise 9: Your code here
# Hint: Use cooks.distance(), car::vif() and lmtest::dwtest()
#       h <- hatvalues(model); cooks_d <- resid(model)^2 / (length(coef(model)) * summary(model)$sigma^2) * h / (1 - h)^2   # equals cooks.distance(model)
# Cook's distance identifies observations that heavily influence the fitted coefficients

# Your solution:


### Reflection Questions

After completing the exercises, consider these questions:

1. **When might linear regression not be appropriate?**
   - Think about non-linear relationships, categorical outcomes, etc.

2. **How do you balance model complexity with interpretability?**
   - Consider the trade-off between adding features and keeping the model simple

3. **What are the key assumptions of linear regression and why do they matter?**
   - Relate each assumption to potential business consequences if violated

4. **How would you explain R² to a non-technical business stakeholder?**
   - Focus on practical interpretation rather than the formula; use the banking model's R² of about 0.01 and the admission model's R² of about 0.64 as your two examples

5. **In what business scenarios would you prefer RMSE over R² as an evaluation metric?**
   - Think about when absolute prediction errors matter more than explained variance

6. **How might you improve the Bitcoin forecasting model, and what is the most likely outcome of trying?**
   - Consider external factors, different features, or alternative approaches, and what market efficiency implies

### Additional Challenges

For further learning:
- Try implementing linear regression from scratch using matrix operations
- Explore regularization techniques (Ridge, Lasso, Elastic Net) in more detail
- Practice with different datasets (housing prices, stock returns, sales forecasting)
- Learn about advanced time series models (ARIMA, GARCH) for financial data
- Investigate non-linear regression techniques (polynomial, spline regression)

### Key Takeaways

- **One regressor, one sentence.** In this run, GRE score alone explained about 64% of the variation in admission chances, and the slope (about 0.01 per GRE point) is the whole story.
- **Only use what is known at decision time.** The banking model uses pre-call features only; `duration`, `y` and `campaign` are excluded. The same rule made us drop everything observed on day *t* from the Bitcoin regressors.
- **A low R² can be the right answer.** The banking model reached a test R² of about 0.01 and an RMSE of about 0.91 that is within one percent of the standard deviation of the target: call length is mostly decided during the call. That is a finding about the business, not a coding error.
- **Check assumptions with the right tool.** Residuals against fitted values for homoscedasticity, a Q-Q plot for normality, VIFs for multicollinearity (the three macro indicators had VIFs of 30 to 65 in this run, so their coefficients cannot be read separately), and Durbin-Watson or the ACF for autocorrelation, but only where the rows have a time order.
- **Compare with the right benchmark.** Price on lagged price gave a test R² of 0.99 that "yesterday's price" matches without any model. On returns, the model's RMSE was 1.003 times the zero forecast and the test R² was -0.006: no forecasting skill, which is what an efficient market looks like. A lag-1 benchmark would have made the same model look 30% better.
- **Significant is not the same as useful.** Two Bitcoin lags had p-values around 0.01 to 0.02 inside an R² of 0.006. Standard errors and p-values say how precisely a coefficient is estimated, not whether it matters.
